# Distilling the Knowledge in a Neural Network: MNIST reproduction

This notebook reproduces the MNIST experiment (Section 3) of Hinton, Vinyals & Dean, [*Distilling the Knowledge in a Neural Network*](https://arxiv.org/abs/1503.02531) (2015):

| Model | Paper test errors (of 10,000) |
|---|---|
| Teacher: 784-1200-1200-10, dropout + weight constraints + jitter | 67 |
| Student: 784-800-800-10, no regularization, true labels | 146 |
| Student: 784-800-800-10, no regularization, distilled from the teacher at T = 20 | 74 |

Running every cell from top to bottom trains the teacher, then both students, then shows a results table and the training curves.

**How to open it.** From the repository root:

```bash
uv sync
uv run --with jupyter jupyter lab mnist/reproduce.ipynb
```

**Run time.** Each run is 3000 epochs. On a GPU that is about 13 min for the teacher, 10 min for the true-label student and 18 min for the distilled student; on a CPU it takes many hours. Set `QUICK = True` below to check that everything works in a few minutes (50 epochs per run; the numbers will then be far from the paper's).

The training code lives in `common.py`, `train_teacher.py` and `train_student.py` next to this notebook; the notebook only calls it, so it trains exactly what the command-line scripts train.

## Setup

In [ ]:
import json
import sys
import time
from pathlib import Path

# Works whether Jupyter was started in the repository root or in mnist/.
HERE = next(p for p in (Path.cwd(), Path.cwd() / "mnist") if (p / "common.py").exists())
sys.path.insert(0, str(HERE))

import matplotlib.pyplot as plt
import torch
from IPython.display import Markdown, display

import train_student
import train_teacher
from common import lr_at, momentum_at

print("torch", torch.__version__, "| CUDA available:", torch.cuda.is_available())

In [ ]:
QUICK = False        # True: QUICK_EPOCHS per run, only to check that the pipeline works
QUICK_EPOCHS = 50
REUSE = True         # skip a run whose saved results match its settings (re-run the plots without retraining)
DEVICE = None        # None: CUDA if available, else CPU

# Checkpoints and results go to mnist/checkpoints/<PREFIX><run>.pt and .json,
# so the notebook never overwrites the command-line runs.
PREFIX = "nb_quick_" if QUICK else "nb_"

## Settings

The paper gives the architectures, the regularizers, the temperature and the results, but no training hyperparameters. For those it points to its reference [5], Hinton et al. (2012), [*Improving neural networks by preventing co-adaptation of feature detectors*](https://arxiv.org/abs/1207.0580), Appendix A. The settings below follow [5] except where noted.

| Setting | Value | Source |
|---|---|---|
| Update rule | Δw = p Δw − (1 − p) ε ∇L, then max-norm projection | [5] |
| Learning rate | ε = lr · 0.998^epoch | [5]; **lr = 1 for the teacher instead of [5]'s 10**, which does not train (the hidden layers die in the first steps) |
| Momentum p | 0.5 → 0.99 linearly over the first 500 epochs | [5] |
| Epochs, batch, init | 3000, 100, weights N(0, 0.01²), biases 0 | [5] |
| Teacher dropout | 0.2 on the input, 0.5 on the hidden units | [5] |
| Teacher max-norm | L2 norm of each hidden unit's incoming weights ≤ 2 | **chosen here**; [5] uses √15 ≈ 3.87 |
| Teacher jitter | shifts of up to 2 pixels in each direction | paper |
| Students | no dropout, no max-norm, no jitter | paper ("no regularization") |
| Student lr | 0.1 for both students | **chosen here**: lr = 1 kills the distilled student, and both students use the same value so that only the targets differ |
| Distillation loss | (1 − w) · T² · CE(teacher at T, student at T) + w · CE(labels, student at T = 1) | paper, Section 2 |
| Temperature T | 20 | paper |
| Hard-label weight w | 0.1 | **chosen here**; the paper only says the hard term gets "a considerably lower weight" |

Change any value here before running the training cells. Set `wandb=True` to also log to Weights & Biases (needs `uv run wandb login` first).

In [ ]:
OPTIM = dict(
    optimizer="momentum", epochs=3000, batch_size=100, lr_decay=0.998,
    momentum_initial=0.5, momentum_final=0.99, momentum_epochs=500,
    init_std=0.01, seed=0, eval_every=25, wandb=False,
)
TEACHER = dict(OPTIM, hidden=1200, bias=True, p_in=0.2, p_hidden=0.5, max_norm=2.0, jitter=2, lr=1.0)
STUDENT = dict(OPTIM, hidden=800, bias=True, p_in=0.0, p_hidden=0.0, max_norm=0, jitter=0, lr=0.1)
STUDENT_HARD = dict(STUDENT, teacher="", temperature=1.0, hard_weight=1.0)
STUDENT_DISTILL = dict(STUDENT, teacher=f"checkpoints/{PREFIX}teacher.pt", temperature=20.0, hard_weight=0.1)

if QUICK:
    for settings in (TEACHER, STUDENT_HARD, STUDENT_DISTILL):
        settings.update(epochs=QUICK_EPOCHS, eval_every=5)

In [ ]:
def run(make_args, train_fn, settings, name):
    # Train one model, or reuse its saved results when REUSE is on and the settings match.
    args = make_args(settings, PREFIX + name, device=DEVICE)
    results = args.out.with_suffix(".json")
    if REUSE and results.exists():
        saved = json.loads(results.read_text())
        if all(saved["config"].get(k) == v for k, v in settings.items()):
            print(f"{name}: reusing {results}")
            return saved["history"]
    start = time.time()
    history = train_fn(args)
    print(f"{name}: {(time.time() - start) / 60:.1f} min")
    return history


histories = {}

## 1. Teacher

In [ ]:
histories["teacher"] = run(train_teacher.make_args, train_teacher.run_teacher, TEACHER, "teacher")

## 2. Students

Both students have the same architecture, initialization and optimizer; only the targets differ. The distilled student reads the teacher trained above and prints its test errors first.

In [ ]:
histories["student_hard"] = run(train_student.make_args, train_student.run_student, STUDENT_HARD, "student_hard")

In [ ]:
histories["student_distill"] = run(train_student.make_args, train_student.run_student, STUDENT_DISTILL, "student_distill")

## 3. Results

In [ ]:
PAPER = {"teacher": 67, "student_hard": 146, "student_distill": 74}

rows = ["| Model | Test errors | Paper | Train errors |", "|---|---|---|---|"]
for name, history in histories.items():
    last = history[-1]
    rows.append(f"| {name} | {last['test_errors']} | {PAPER[name]} | {last.get('train_errors', '')} |")
if {"student_hard", "student_distill"} <= histories.keys():
    ours = histories["student_distill"][-1]["test_errors"] / histories["student_hard"][-1]["test_errors"]
    rows.append(f"\nDistilled / true-label student errors: **{ours:.2f}** (paper: {74 / 146:.2f})")
display(Markdown("\n".join(rows)))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
for (name, history), color in zip(histories.items(), ("C0", "C1", "C2")):
    ax.plot([h["epoch"] for h in history], [h["test_errors"] for h in history], color=color, label=name)
    ax.axhline(PAPER[name], color=color, linestyle="--", linewidth=1)
ax.set(xlabel="epoch", ylabel="test errors (of 10,000)", yscale="log", title="Test errors (dashed: paper)")
ax.legend()
plt.show()

In [ ]:
# Learning rate and momentum schedules (the same rule for every run, with different lr).
fig, (ax_lr, ax_mom) = plt.subplots(1, 2, figsize=(11, 4))
for name, make_args, settings in (
    ("teacher", train_teacher.make_args, TEACHER),
    ("students", train_student.make_args, STUDENT_HARD),
):
    args = make_args(settings, "schedule")
    epochs = range(args.epochs)
    ax_lr.plot(epochs, [lr_at(args, e) for e in epochs], label=name)
    if args.optimizer == "momentum":
        ax_mom.plot(epochs, [momentum_at(args, e) for e in epochs], label=name)
ax_lr.set(xlabel="epoch", ylabel="ε", yscale="log", title="Learning rate")
ax_mom.set(xlabel="epoch", ylabel="p", title="Momentum")
ax_lr.legend()
ax_mom.legend()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for name in ("teacher", "student_hard"):
    if name in histories:
        h = histories[name]
        axes[0].plot([e["epoch"] for e in h], [e["train_loss"] for e in h], label=name)
axes[0].set(xlabel="epoch", ylabel="cross-entropy", yscale="log", title="Training loss")
axes[0].legend()

if "student_distill" in histories:
    h = histories["student_distill"]
    axes[1].plot([e["epoch"] for e in h], [e["train_soft_kl"] for e in h])
axes[1].set(xlabel="epoch", ylabel="KL(teacher ‖ student) at T", yscale="log", title="Distilled student: soft-target KL")

for name in ("student_hard", "student_distill"):
    if name in histories:
        h = histories[name]
        line, = axes[2].plot([e["epoch"] for e in h], [e["test_errors"] for e in h], label=f"{name} test")
        axes[2].plot([e["epoch"] for e in h], [max(e["train_errors"], 0.5) for e in h], color=line.get_color(), linestyle=":", label=f"{name} train")
axes[2].set(xlabel="epoch", ylabel="errors", yscale="log", title="Students: train (dotted) vs test errors")
axes[2].legend(fontsize=8)
plt.tight_layout()
plt.show()

## Notes

- **Reading the losses.** The training loss of the teacher is measured with dropout and jitter on, so it stays above 0. The true-label student reaches a training loss of about 0, with 0 training errors: it memorizes the training set, and its test errors stop changing. The distilled student's loss contains the entropy of the teacher's soft targets, which the student cannot reduce (about 0.9 · 400 · 2.16 ≈ 777 at T = 20), so the plot shows only the KL part, whose minimum is 0.
- **Our results with these settings** (one run each, on a GPU): teacher 66 test errors, true-label student 177, distilled student about 85. Distillation roughly halves the student's errors, as in the paper (74 / 146 = 0.51). The absolute numbers differ from the paper's because the student's training settings are not given there.
- **Run-to-run noise.** Repeating a run with another `seed` changes the test errors by a few; compare differences larger than that.
- **No validation set.** Every model is trained on all 60,000 training images and evaluated on the 10,000 test images, as in the paper. The settings marked "chosen here" were picked by looking at test errors, so they are not a tuned, held-out result.